# **Selecting Models**

**Overivew**: This script compares the performance of 8 models to evaluate how error-prone each model is. The dataset used for this model comparison is created in another notebook:
- *data_processing.ipynb*

**Outputs** (saved under `Output_data_from_model_selection_scripts`)
- `model_performance_metrics.csv`

**Processing Steps**
1. Ordinary Least Squares (OLS)
2. K-Nearest Neighbors (KNN)
3. Support Vector Machines (SVM)
4. Multi-Layer Perceptron (MLP)
5. Random Foest (RF)
6. XGBoost
7. CatBoost 
8. LightGBM (LGBM)


**Start date:** *Oct 23, 2024*<br>
**Last update:** *Mar 24, 2025*<br>
**Authors:** *Benjamin Tarver & Kanaha Shoji* <br>

In [4]:
# Import necessary packages
import pandas as pd
import numpy as np
import multiprocessing
import os
import random

# Scaling/sampling data
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

# ML models + optimization
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import Pool, CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # root_mean_squared_error,

In [5]:
# Set base directory
base_dir = '/Users/shoji/Library/CloudStorage/OneDrive-epfl.ch/2025_03_Melbourne_walkability_study_final'
# Set directory for output from data pocessing for modeling scripts
processed_dir = os.path.join(base_dir,'Output_data_from_processing_scripts')
# Set directory for output data
output_dir = os.path.join(base_dir,'Output_data_from_model_selection_scripts')

In [6]:
# Import processed feature data
processed_melbourne_data = pd.read_csv(os.path.join(processed_dir,'processed_melbourne_data_for_modeling.csv'))

In [7]:
# Set random seed for reproducibility
RandomSeed = 0
random.seed(RandomSeed)

In [8]:
# Replace the Location_ID values with randomized identifiers, 
# ensuring that each location still maintains a consistent, unique ID but without any inherent bias due to the numerical patterns.
# Generate a random number for each unique Location_ID
location_map = {location: np.random.randint(1, 10000) for location in processed_melbourne_data['Location_id'].unique()}
processed_melbourne_data['Location_id'] = processed_melbourne_data['Location_id'].map(location_map)

In [9]:
# Specify the number of samples you want
n_samples = int(len(processed_melbourne_data) / 100)

# Create the StratifiedShuffleSplit object
split = StratifiedShuffleSplit(test_size=n_samples, random_state=RandomSeed)

# Perform the split (since StratifiedShuffleSplit returns indices, we use them to index the DataFrame)
for train_index, test_index in split.split(processed_melbourne_data, processed_melbourne_data['Year']):
    resampled_df = processed_melbourne_data.iloc[test_index]

In [10]:
# FOR DEMONSTRATIVE PURPOSES
# Generate statistics on resampled data to find any potential biases
special_days_prop = len(resampled_df.loc[resampled_df['Day_type'] == 1]) / len(resampled_df)
hours_dist = [len(resampled_df.loc[resampled_df['Hour'] == hr]) for hr in resampled_df['Hour'].unique()]
years_dist = resampled_df['Year'].value_counts().to_list()
sensors_dist = resampled_df['Location_id'].value_counts().to_list()

print(f"Proportion of special days to normal (expected =~0.3): {special_days_prop}")
print(f"Number of readings selected for each hour (0-23) from data (not ordered): \n{hours_dist}")
print(f"Number of readings selected for each year (2009-2019) from data in reverse order of year: \n{years_dist}")

Proportion of special days to normal (expected =~0.3): 0.3161025641025641
Number of readings selected for each hour (0-23) from data (not ordered): 
[1012, 952, 1001, 1052, 969, 1115, 1017, 966, 1061, 1001, 1024, 1059, 967, 1018, 1001, 1067, 997, 985, 1032, 946, 1057, 1087, 993, 996]
Number of readings selected for each year (2009-2019) from data in reverse order of year: 
[3984, 3759, 3075, 2884, 2640, 2166, 1472, 1230, 1214, 1137, 814]


In [11]:
# Declare feature and target variables and perform train-test split
y = resampled_df['Pedestrian_count']
x = resampled_df.drop(columns='Pedestrian_count')
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=RandomSeed)

In [12]:
# FOR DEMONSTRATIVE PURPOSES ONLY
# Print statistics about pedestrian counts
print(f"Mean: {np.mean(y)}")
print(f"Median: {np.median(y)}")
print(f"Stdev: {np.std(y)}")
print(f"Range: {np.max(y)-np.min(y)}")

Mean: 584.196841025641
Median: 245.0
Stdev: 798.2905734465422
Range: 5877.0


In [13]:
# Perform feature scaling for models which require such an operation, then perform train-test split
x_scaled = pd.DataFrame(StandardScaler().fit_transform(x), columns=x.columns)
y_scaled = StandardScaler().fit_transform(y.to_frame())
x_s_train, x_s_test, y_s_train, y_s_test = train_test_split(x_scaled, y_scaled, test_size=0.25, random_state=RandomSeed)

In [14]:
# Initialize an empty list to store results
results = []

## Ordinary Least Squares (OLS)

In [53]:
# Spin up and run OLS model
ols = LinearRegression()
ols.fit(x_s_train, y_train)
y_pred = ols.predict(x_s_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")
# Append metrics to results
results.append({
    "Model": "Ordinary Least Squares",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

MAE: 441.2364474962413
MSE: 405339.6792749468
RMSE: 636.662924375958
R²: 0.370304131464512


## K-Nearest Neighbors (KNN)

In [14]:
# Spin up and run kNN model
knn = KNeighborsRegressor() 
clf = GridSearchCV(knn, {"n_neighbors": [4, 6, 8], "weights": ['distance']}, n_jobs=multiprocessing.cpu_count() // 2)
clf.fit(x_s_train, y_train)
best_params = clf.best_params_
print(clf.best_score_)
print(best_params)

best_knn = KNeighborsRegressor(n_neighbors=best_params['n_neighbors'], weights=best_params['weights'])
best_knn.fit(x_s_train, y_train)
y_pred = best_knn.predict(x_s_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")

# Append metrics to results
results.append({
    "Model": "k-Nearest Neighbors",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

0.5510064719574302
{'n_neighbors': 8, 'weights': 'distance'}
MAE: 330.70527479103134
MSE: 274355.0691874014
RMSE: 523.7891457327094
R²: 0.5737889419360565


## Support Vector Machines (SVM)

In [55]:
# Spin up and run SVM model
svm = SVR(kernel='rbf') # somehow the other kernels are even worse
# clf = GridSearchCV(svm, {"epsilon": [0.005, 0.01, 0.05, 0.1, 0.5], "C": [100, 500, 1000]}, n_jobs=multiprocessing.cpu_count() // 2) # .653
# clf.fit(x_s_train, y_train)
# best_params = clf.best_params_
# print(clf.best_score_)
# print(best_params)

best_svm = SVR(kernel='rbf', C=10000, epsilon=0.01)

best_svm.fit(x_s_train, y_train)
y_pred = best_svm.predict(x_s_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")
# Append metrics to results
results.append({
    "Model": "Support Vector Machine",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

MAE: 271.9776413062
MSE: 225328.4343092178
RMSE: 474.6877229392159
R²: 0.6499519010774173


## Multi-Layer Perceptron (MLP)

In [15]:
# Spin up and run neural network model (Multi-Layer Perceptron)
# mlp = MLPRegressor(random_state=RandomSeed)
# clf = GridSearchCV(mlp, {"max_iter": [100, 300, 1000], "alpha": [0.00001, 0.0001, 0.001], "solver": ['lbfgs', 'sgd', 'adam']}, n_jobs=multiprocessing.cpu_count() // 2)
# # {'alpha': 0.0001, 'max_iter': 1000, 'solver': 'lbfgs'}
# clf.fit(x_s_train, y_s_train)
# best_params = clf.best_params_
# print(clf.best_score_)
# print(best_params)

# Use best parameters to train and fit model
# best_mlp = MLPRegressor(random_state=RandomSeed, max_iter=best_params['max_iter'], alpha=best_params['alpha'], solver=best_params['solver'])

# Suppress convergence warning
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
# Convergence warning was suppressed: The lbfgs solver did not converge even when increased number of iterations, and
#                                        their performance was worse than the default parameters
# The result with max_iter = 10000
# MAE: 194.45071085097308
# MSE: 91745.45913380895
# RMSE: 302.89512893707774
# R²: 0.8574732760513593

best_mlp = MLPRegressor(random_state=RandomSeed, max_iter=1000, alpha=0.0001, solver='lbfgs')
best_mlp.fit(x_s_train, y_train)
y_pred = best_mlp.predict(x_s_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")
# Append metrics to results
results.append({
    "Model": "Multi-Layer Perceptron",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

MAE: 188.1670719177911
MSE: 90727.37671098839
RMSE: 301.2098549367009
R²: 0.8590548687950685


## Random Forest (RF)

In [57]:
# Spin up and run RF model
rf = RandomForestRegressor(n_jobs=multiprocessing.cpu_count() // 2, random_state=RandomSeed)
# clf = GridSearchCV(rf, {"max_depth": [8, 16], "n_estimators": [250, 500, 1000]}, n_jobs=multiprocessing.cpu_count() // 2)
# clf.fit(x_train, y_train)
# best_params = clf.best_params_
# print(clf.best_score_)
# print(best_params)

# best_rf = RandomForestRegressor(random_state=RandomSeed, n_estimators=best_params['n_estimators'], max_depth=best_params['max_depth'])
best_rf = RandomForestRegressor(random_state=RandomSeed, n_estimators=1000, max_depth=16)
best_rf.fit(x_train, y_train.to_list())
y_pred = best_rf.predict(x_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")

# Append metrics to results
results.append({
    "Model": "Random Forest",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

MAE: 133.86769252529552
MSE: 67235.52444199786
RMSE: 259.29813813831726
R²: 0.8955495005184909


## XGBoost

In [58]:
# Spin up and run XGB model
xgb = XGBRegressor(n_jobs=multiprocessing.cpu_count() // 2, tree_method="hist", random_state=RandomSeed)
# clf = GridSearchCV(xgb, {"max_depth": [2, 4, 8, 16], "n_estimators": [50, 100, 250, 500]}, verbose=1, n_jobs=multiprocessing.cpu_count() // 2)
# clf = GridSearchCV(xgb, {"max_depth": [7, 8, 9], "n_estimators": [40, 50, 60]}, verbose=1, n_jobs=multiprocessing.cpu_count() // 2)
# clf.fit(x_train, y_train)
# best_params = clf.best_params_
# print(clf.best_score_)
# print(best_params)

# Use best parameters to train and fit model
# best_xgb = XGBRegressor(n_jobs=multiprocessing.cpu_count() // 2, tree_method="hist", max_depth=best_params['max_depth'], n_estimators=best_params['n_estimators'], random_state=0)
best_xgb = XGBRegressor(n_jobs=multiprocessing.cpu_count() // 2, tree_method="hist", max_depth=8, n_estimators=50, random_state=RandomSeed)
best_xgb.fit(x_train, y_train)
y_pred = best_xgb.predict(x_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")

# Append metrics to results
results.append({
    "Model": "XGBoost",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

MAE: 136.50931082234095
MSE: 64229.01334508041
RMSE: 253.43443598903528
R²: 0.9002201205274214


## CatBoost

In [59]:
# Run CatBoostRegressor (gradient-boosted trees model, does not require feature scaling)
# initialize Pool
train_pool = Pool(x_train, y_train)
test_pool = Pool(x_test)

# specify the training parameters; note that iterations = tree count
cat = CatBoostRegressor(random_state=RandomSeed, silent=True)
clf = GridSearchCV(cat, {"learning_rate":[0.1], "max_depth": [2,4,6,8], "iterations": [150, 1500]}) # 0.893 for 1k
clf.fit(x_train, y_train)
best_params = clf.best_params_
print(clf.best_score_)
print(best_params)

# train the model & predict
best_cat = CatBoostRegressor(random_state=RandomSeed, learning_rate=best_params['learning_rate'], iterations=best_params['iterations'], silent=True)
# best_cat = CatBoostRegressor(random_state=RandomSeed, learning_rate=0.1, iterations=1500, silent=True)
best_cat.fit(train_pool)
y_pred = best_cat.predict(test_pool)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")

# Append metrics to results
results.append({
    "Model": "CatBoost",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

0.904933233530391
{'iterations': 1500, 'learning_rate': 0.1, 'max_depth': 8}
MAE: 136.81218091892515
MSE: 59115.824527365185
RMSE: 243.13746014829798
R²: 0.9081634678930592


## LightGBM (LGBM)

In [60]:
# Spin up and run LGB model
lgb = LGBMRegressor(n_jobs=multiprocessing.cpu_count() // 2, boosting_type='gbdt', random_state=RandomSeed)
# clf = GridSearchCV(lgb, {"n_estimators": [250, 500, 1000], "learning_rate": [0.03, 0.1, 0.3]}, n_jobs=multiprocessing.cpu_count() // 2)
# clf = GridSearchCV(lgb, {"n_estimators": [2000, 2500, 3000], "learning_rate": [0.02]}, n_jobs=multiprocessing.cpu_count() // 2)
# clf.fit(x_train, y_train)
# best_params = clf.best_params_

# Use best parameters to train and fit model
# best_lgb = LGBMRegressor(n_jobs=multiprocessing.cpu_count() // 2, boosting_type='gbdt', learning_rate=best_params['learning_rate'], n_estimators=best_params['n_estimators'], random_state=0)
best_lgb = LGBMRegressor(n_jobs=multiprocessing.cpu_count() // 2, boosting_type='gbdt', learning_rate=.02, n_estimators=2500, random_state=RandomSeed)
best_lgb.fit(x_train, y_train)
y_pred = best_lgb.predict(x_test)

# Example performance metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R²: {r2}")
# print(clf.best_score_)
# print(best_params)
# Append metrics to results
results.append({
    "Model": "LGBoost",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2
})

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005417 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3698
[LightGBM] [Info] Number of data points in the train set: 18281, number of used features: 50
[LightGBM] [Info] Start training from score 582.459931
MAE: 130.3481707195726
MSE: 57976.05204124151
RMSE: 240.78216719940352
R²: 0.9099341063532916


In [23]:
df = pd.DataFrame(results)
print(df)

                    Model         MAE            MSE        RMSE        R2
0  Ordinary Least Squares  441.236447  405339.679275  636.662924  0.370304
1     k-Nearest Neighbors  330.038025  274087.410750  523.533581  0.574205
2  Support Vector Machine  271.977641  225328.434309  474.687723  0.649952
3  Multi-Layer Perceptron  188.167072   90727.376711  301.209855  0.859055
4           Random Forest  133.867693   67235.524442  259.298138  0.895550
5                 XGBoost  136.509311   64229.013345  253.434436  0.900220
6                CatBoost  136.812181   59115.824527  243.137460  0.908163
7                 LGBoost  130.348171   57976.052041  240.782167  0.909934


In [62]:
df.to_csv(os.path.join(output_dir, 'model_performance_metrics.csv'), index=False)